# Lecture 5 Demo: The Photon Transfer Curve

**Instructor demo.** About twenty seconds, no external files.

We derived on the board that

$$\mathrm{Var}[D] = K\,(\mathbb{E}[D] - B) + K^2\sigma_r^2$$

so variance plotted against mean is a straight line whose slope is the conversion gain
and whose intercept gives the read noise.

Here we take photographs of a simulated blank wall and recover the gain and the read
noise of a sensor we never opened. Then we check the answers against the truth, because
the simulation knows them.

In [3]:
import sys
from pathlib import Path

# Put the shared slide helpers on the path, wherever this notebook is opened from.
for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (parent / "psu_imaging.py").exists():
        sys.path.insert(0, str(parent))
        break

import cv2
import matplotlib.pyplot as plt
import numpy as np
import psu_imaging as psu

plt.rcParams.update(psu.figure_style())
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(513)
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

OpenCV 4.13.0 | NumPy 2.4.3


In [ ]:
# Ground truth, known only because this is a simulation.
TRUE_K = (psu.WHITE_LEVEL - psu.BLACK_LEVEL) / psu.FULL_WELL     # DN per electron
TRUE_READ = psu.READ_NOISE                                        # electrons RMS

print(f"true conversion gain K   {TRUE_K:.4f} DN/e-")
print(f"true read noise          {TRUE_READ:.2f} e-")
print(f"true full well           {psu.FULL_WELL:.0f} e-")
print(f"black level              {psu.BLACK_LEVEL} DN, white level {psu.WHITE_LEVEL} DN")
print("\nEverything below is measured from flat fields only.")

## 1. Capture flat fields

A flat field is a photograph of a uniformly lit blank surface: defocused white wall,
integrating sphere, or a tablet showing a gray card. Two frames at each level.

We add a small photo-response non-uniformity so the sensor has realistic fixed-pattern
structure. That will matter in section 2.

In [ ]:
PRNU = 0.012          # 1.2 percent per-pixel gain variation, fixed for this sensor

def flat_pair(electrons, level_index):
    """Two independent frames at the same illumination."""
    patch = np.full((128, 128), electrons / psu.FULL_WELL)
    a = psu.expose_and_digitize(patch, np.random.default_rng(2 * level_index), prnu=PRNU)
    b = psu.expose_and_digitize(patch, np.random.default_rng(2 * level_index + 1), prnu=PRNU)
    return a.astype(float), b.astype(float)


levels = np.logspace(np.log10(5), np.log10(psu.FULL_WELL * 0.92), 22)
pairs = [flat_pair(e, i) for i, e in enumerate(levels)]
print(f"{len(pairs)} exposure levels, two frames each, 128 x 128 px")

fig, axes = plt.subplots(1, 4, figsize=(11, 2.9))
for ax, i in zip(axes, [0, 7, 14, 21]):
    a, _ = pairs[i]
    ax.imshow(a, cmap="gray")
    ax.set_title(f"{levels[i]:.0f} e$^-$", fontsize=9.5)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. The wrong way, and why it is wrong

The obvious thing is to take the variance of a **single** flat field. But a single frame
contains fixed-pattern noise as well as random noise, and fixed-pattern noise is not part
of the model we derived. It inflates the variance and biases the gain.

The fix is to use the **difference of two frames**. The fixed pattern is identical in
both, so subtracting removes it; the random parts are independent, so the variance
doubles and we halve it back.

In [ ]:
means, var_single, var_pair = [], [], []
for a, b in pairs:
    means.append((a.mean() + b.mean()) / 2 - psu.BLACK_LEVEL)
    var_single.append(a.var())
    var_pair.append(np.var(a - b) / 2)

means = np.array(means); var_single = np.array(var_single); var_pair = np.array(var_pair)

def fit_ptc(mean, var, frac=0.7):
    """Fit the linear region, below `frac` of saturation."""
    ok = mean < (psu.WHITE_LEVEL - psu.BLACK_LEVEL) * frac
    slope, intercept = np.polyfit(mean[ok], var[ok], 1)
    return slope, intercept, ok

for label, v in (("single frame", var_single), ("difference of pairs", var_pair)):
    K, c, _ = fit_ptc(means, v)
    read = np.sqrt(max(c, 0)) / K
    print(f"{label:22s}  K = {K:.4f} DN/e-  ({(K - TRUE_K) / TRUE_K * +100:+5.1f} %)   "
          f"read noise = {read:5.2f} e-")
print(f"{'truth':22s}  K = {TRUE_K:.4f} DN/e-            read noise = {TRUE_READ:5.2f} e-")

## 3. Fit the line

Slope is the conversion gain. Intercept is $K^2\sigma_r^2$, so the read noise in electrons
is $\sqrt{\text{intercept}}/K$.

Fit only the linear region. Near saturation the variance bends over, because clipping
truncates the distribution, and including those points drags the slope down.

In [ ]:
K_hat, c_hat, used = fit_ptc(means, var_pair)
read_hat = np.sqrt(max(c_hat, 0)) / K_hat
full_well_hat = (psu.WHITE_LEVEL - psu.BLACK_LEVEL) / K_hat
dr_stops = np.log2(full_well_hat / read_hat)

print(f"conversion gain   {K_hat:.4f} DN/e-     (true {TRUE_K:.4f})")
print(f"read noise        {read_hat:.2f} e-        (true {TRUE_READ:.2f})")
print(f"full well         {full_well_hat:.0f} e-       (true {psu.FULL_WELL:.0f})")
print(f"dynamic range     {dr_stops:.1f} stops")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7))
xs = np.linspace(0, means.max() * 1.02, 100)

axes[0].plot(means[used], var_pair[used], "o", ms=6, color=psu.PSU_BLUE, label="fitted")
axes[0].plot(means[~used], var_pair[~used], "o", ms=6, mfc="none",
             color=psu.PSU_GRAY, label="excluded (near saturation)")
axes[0].plot(xs, K_hat * xs + c_hat, "-", lw=2, color=psu.PSU_FOREST,
             label=f"slope {K_hat:.4f}, intercept {c_hat:.1f}")
axes[0].set_xlabel("mean above black (DN)"); axes[0].set_ylabel("variance (DN$^2$)")
axes[0].set_title("Photon transfer curve"); axes[0].legend(fontsize=8.5)

axes[1].loglog(means, var_pair, "o", ms=6, color=psu.PSU_BLUE)
axes[1].loglog(xs[1:], K_hat * xs[1:], "--", lw=1.5, color=psu.PSU_GREEN,
               label="shot term, slope 1")
axes[1].axhline(c_hat, ls=":", lw=1.5, color=psu.PSU_WARM, label="read-noise floor")
axes[1].set_xlabel("mean above black (DN)"); axes[1].set_ylabel("variance (DN$^2$)")
axes[1].set_title("Log axes: the two terms separate"); axes[1].legend(fontsize=8.5)
plt.tight_layout(); plt.show()

## 4. Predict the SNR curve

We now have every parameter in the model, so we can predict SNR at any signal level and
check the prediction against a direct measurement.

In [ ]:
snr_measured = means / np.sqrt(var_pair)
e_axis = means / K_hat
snr_model = e_axis / np.sqrt(e_axis + read_hat ** 2)

plt.figure(figsize=(7, 3.6))
plt.loglog(e_axis, snr_measured, "o", ms=6, color=psu.PSU_BLUE, label="measured")
plt.loglog(e_axis, snr_model, "-", lw=2, color=psu.PSU_FOREST, label="predicted from K and $\\sigma_r$")
plt.loglog(e_axis, np.sqrt(e_axis), "--", lw=1.3, color=psu.PSU_GREEN, label="photon limit")
plt.axvline(read_hat ** 2, color="#999", lw=1.2)
plt.text(read_hat ** 2 * 1.3, 0.6, "knee", fontsize=9, color="#555")
plt.xlabel("signal (electrons)"); plt.ylabel("SNR"); plt.legend(fontsize=9)
plt.title("The model predicts the measurement")
plt.tight_layout(); plt.show()

err = np.abs(snr_measured - snr_model) / snr_model
print(f"median relative error between prediction and measurement: {np.median(err) * 100:.1f} %")

## 5. Fixed-pattern noise does not average away

Stack many frames at one level. The random part falls as $1/\sqrt{n}$. The fixed part is
the same in every frame, so it simply stays, and the curve flattens onto a floor.

In [ ]:
LEVEL_E = 4000.0
patch = np.full((128, 128), LEVEL_E / psu.FULL_WELL)
frames = [psu.expose_and_digitize(patch, np.random.default_rng(500 + k), prnu=PRNU).astype(float)
          for k in range(64)]

ns = [1, 2, 4, 8, 16, 32, 64]
sds = [np.mean(frames[:n], axis=0).std() for n in ns]

predicted_floor = PRNU * LEVEL_E * K_hat        # fixed gain variation, in DN
random_1 = np.sqrt(LEVEL_E + read_hat ** 2) * K_hat

plt.figure(figsize=(7, 3.4))
plt.loglog(ns, sds, "o-", ms=6, lw=1.8, color=psu.PSU_BLUE, label="measured spatial sd")
plt.loglog(ns, random_1 / np.sqrt(ns), "--", lw=1.4, color=psu.PSU_GREEN,
           label=r"random only, $1/\sqrt{n}$")
plt.axhline(predicted_floor, ls=":", lw=1.6, color=psu.PSU_WARM,
            label=f"PRNU floor, {predicted_floor:.1f} DN")
plt.xlabel("frames averaged"); plt.ylabel("spatial standard deviation (DN)")
plt.legend(fontsize=9); plt.title("Averaging stops helping once you hit the fixed pattern")
plt.tight_layout(); plt.show()

print(f"1 frame:  {sds[0]:.1f} DN")
print(f"64 frames: {sds[-1]:.1f} DN, against a predicted PRNU floor of {predicted_floor:.1f} DN")
print("\nTo go below the floor you must calibrate it out with a flat field, not average harder.")

## 6. Your own camera

Shoot flat fields: defocus completely on an evenly lit blank wall, or photograph a tablet
showing a uniform gray. Vary exposure time over a wide range, two frames at each setting,
raw, fixed ISO, no auto anything.

Drop them in `Images/` and this cell will run the same fit on one green plane.

In [ ]:
raws = psu.find_raw_files()

if len(raws) < 4:
    print(f"Found {len(raws)} raw file(s). Need at least 4 flat fields (two levels, two frames).")
    print("The simulation above stands on its own; this cell is for your own captures.")
else:
    import rawpy

    def green_plane(path):
        with rawpy.imread(str(path)) as raw:
            bayer = raw.raw_image_visible.astype(float)
            black = float(np.mean(raw.black_level_per_channel))
        return bayer[0::2, 1::2], black          # green on the red rows of an RGGB tile

    planes, blacks = zip(*[green_plane(p) for p in raws])
    order = np.argsort([p.mean() for p in planes])
    planes = [planes[i] for i in order]
    black = float(np.mean(blacks))

    m, v = [], []
    for a, b in zip(planes[0::2], planes[1::2]):
        n = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
        a, b = a[:n[0], :n[1]], b[:n[0], :n[1]]
        m.append((a.mean() + b.mean()) / 2 - black)
        v.append(np.var(a - b) / 2)
    m, v = np.array(m), np.array(v)

    if len(m) >= 2:
        K, c = np.polyfit(m, v, 1)
        print(f"from {len(raws)} files: K = {K:.4f} DN/e-, "
              f"read noise = {np.sqrt(max(c, 0)) / K:.2f} e-")
        plt.figure(figsize=(6, 3.2))
        plt.plot(m, v, "o", color=psu.PSU_BLUE)
        plt.plot(m, K * m + c, "-", color=psu.PSU_FOREST)
        plt.xlabel("mean above black (DN)"); plt.ylabel("variance (DN$^2$)")
        plt.title("Your sensor"); plt.tight_layout(); plt.show()
    else:
        print("Need pairs at several different exposure levels to fit a line.")

## Back to the slides

Land on this: we photographed a blank wall and recovered the conversion gain, the read
noise, the full well, and the dynamic range of a chip we never opened. The only thing that
made it possible was one line of algebra that said variance is affine in the mean.

That line is also the reason Project 2's learned denoiser works: you will generate its
training pairs from exactly this model, fitted to your own camera.